# Week 3 실습 — MLOps Infra: Code & Data Versioning

이 노트북은 **Git + DVC 기반 코드/데이터 버전관리 실습**을 위한 가이드입니다.  
목표는 명령어를 외우는 것이 아니라, **재현 가능한 ML 실험 구조**를 직접 만들어보는 것입니다.

## 이번 실습에서 만들 산출물

- 로컬 Git 저장소
- 원격 Git 저장소(GitHub 등)
- `README.md`, `.gitignore`
- 초기 commit / push 이력
- DVC 초기화 구조(`.dvc/`)
- 데이터 추적용 `.dvc` 메타파일
- DVC remote 연결
- 데이터 버전 변경 및 복원 경험
- 제출용 회고 포인트 정리

## 권장 진행 방식

1. 마크다운 설명을 먼저 읽습니다.
2. 코드 셀은 **한 셀씩 순서대로** 실행합니다.
3. 명령어 실행 후 출력 로그를 꼭 확인합니다.
4. 실습 중 오류가 나면 마지막의 **Troubleshooting** 섹션을 참고합니다.

## 실습 전 준비 사항

* **Python 3.10 이상**
* **Git 설치**
* **uv (Python 패키지 및 환경 관리 도구)**
* **DVC (Data Version Control)**
* **GitHub 계정** (또는 Git 원격 저장소)
* **터미널 사용 가능 환경**

---

## 환경 및 패키지 설치

실습에서는 **uv 기반 Python 환경 관리**를 사용합니다.

### 1. 기본 환경 확인

```bash
git --version
python --version
uv --version
```

---

### 2. 실습 패키지 설치

```bash
uv pip install dvc pandas numpy
```

---

### 3. 설치 확인

```bash
python -m dvc --version
python -m pip --version
```

---

💡 **MLOps 실습 환경에서는**

* Git → **Code Versioning**
* DVC → **Data Versioning**
* uv → **Python Environment Management**

이 세 가지를 함께 사용하여 **재현 가능한 머신러닝 프로젝트 환경**을 구성합니다.

In [1]:
# 환경 및 버전 확인
!git --version
!uv --version
!python --version

# 패키지 설치 및 실행
!uv pip install -U pip dvc pandas numpy
!python -m pip --version
!python -m dvc --version

git version 2.50.1 (Apple Git-155)
uv 0.10.7 (Homebrew 2026-02-27)
Python 3.12.12
Resolved 104 packages in 256ms                                       
⠙ Preparing packages... (0/1)                                                   
⠙ Preparing packages... (0/1)-------------------     0 B/288.24 KiB          
⠙ Preparing packages... (0/1)------------------- 16.00 KiB/288.24 KiB        
⠙ Preparing packages... (0/1)------------------- 32.00 KiB/288.24 KiB        
⠙ Preparing packages... (0/1)------------------- 48.00 KiB/288.24 KiB        
⠙ Preparing packages... (0/1)------------------- 64.00 KiB/288.24 KiB        
⠙ Preparing packages... (0/1)------------------- 80.00 KiB/288.24 KiB        
⠙ Preparing packages... (0/1)------------------- 96.00 KiB/288.24 KiB        
⠙ Preparing packages... (0/1)------------------- 112.00 KiB/288.24 KiB       
⠙ Preparing packages... (0/1)2m----------------- 128.00 KiB/288.24 KiB       
⠙ Preparing packages... (0/1)---------------- 144.00 KiB/288.24 K

## 사용하고 있는 폴더로 위치 바꿀 것.

In [2]:
import os

# 사용하고 있는 폴더로 위치 바꿀 것.
target_dir = "/Users/sungjae-cha/Documents/03 아주대AI대학원/2026-1st-semester/ajou-mlops-2026-1nd-semester"
os.chdir(target_dir)

print("Changed CWD:", os.getcwd())

Changed CWD: /Users/sungjae-cha/Documents/03 아주대AI대학원/2026-1st-semester/ajou-mlops-2026-1nd-semester


In [3]:
import os, sys, subprocess

print("Python:", sys.executable)
print("CWD:", os.getcwd())
subprocess.run("pwd", shell=True)
subprocess.run("which python", shell=True)
subprocess.run("which git", shell=True)
subprocess.run("git rev-parse --show-toplevel", shell=True)

Python: /Users/sungjae-cha/Documents/03 아주대AI대학원/2026-1st-semester/ajou-mlops-2026-1nd-semester/.venv/bin/python
CWD: /Users/sungjae-cha/Documents/03 아주대AI대학원/2026-1st-semester/ajou-mlops-2026-1nd-semester
/Users/sungjae-cha/Documents/03 아주대AI대학원/2026-1st-semester/ajou-mlops-2026-1nd-semester
/Users/sungjae-cha/Documents/03 아주대AI대학원/2026-1st-semester/ajou-mlops-2026-1nd-semester/.venv/bin/python
/usr/bin/git
/Users/sungjae-cha/Documents/03 아주대AI대학원/2026-1st-semester/ajou-mlops-2026-1nd-semester


CompletedProcess(args='git rev-parse --show-toplevel', returncode=0)

In [4]:
import sys, subprocess, platform
print("Python:", sys.version.split()[0])
print("OS:", platform.platform())

def run(cmd):
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    return result.returncode

run("git --version")
run("dvc --version")

Python: 3.12.12
OS: macOS-26.3.1-arm64-arm-64bit
$ git --version
git version 2.50.1 (Apple Git-155)

$ dvc --version
3.67.0



0

## 실습 시나리오

이번 실습에서는 다음 흐름을 따릅니다.

1. 프로젝트 폴더 생성
2. Git 저장소 초기화
3. `README.md`, `.gitignore` 작성
4. 첫 commit 생성
5. 샘플 데이터 생성
6. DVC 초기화 및 데이터 추적
7. DVC remote 연결
8. 데이터 버전 변경
9. Git commit + DVC push
10. 과거 버전으로 복원

## 오늘 핵심 개념

- **Git**: 코드와 메타정보의 히스토리 관리
- **DVC**: 실제 데이터와 실험 입력의 버전 관리
- **재현성(Reproducibility)**: 특정 시점의 코드 + 데이터 상태를 다시 되살릴 수 있는 것

## 폴더 구조(권장)

```text
week3-mlops-versioning/
├── .dvc/
├── .gitignore
├── README.md
├── data/
│   └── raw/
│       └── customer_churn_sample.csv
├── notebooks/
├── reports/
└── src/
```

In [5]:
from pathlib import Path

PROJECT_NAME = "week3-mlops-versioning"
root = Path.cwd() / PROJECT_NAME
for p in [root, root/"data"/"raw", root/"notebooks", root/"reports", root/"src"]:
    p.mkdir(parents=True, exist_ok=True)

print("Created project at:", root)
for path in sorted(root.rglob("*")):
    print(path.relative_to(root))

Created project at: /Users/sungjae-cha/Documents/03 아주대AI대학원/2026-1st-semester/ajou-mlops-2026-1nd-semester/week3-mlops-versioning
.dvc
.dvc/.gitignore
.dvc/cache
.dvc/cache/files
.dvc/cache/files/md5
.dvc/cache/files/md5/00
.dvc/cache/files/md5/00/35e53b22bf409adda6b79737217bff
.dvc/cache/files/md5/79
.dvc/cache/files/md5/79/1469f39c8c084e9b1f532f12a4ac0e
.dvc/config
.dvc/tmp
.dvc/tmp/btime
.dvc/tmp/lock
.dvc/tmp/rwlock
.dvc/tmp/rwlock.lock
.dvcignore
.git
.git/COMMIT_EDITMSG
.git/HEAD
.git/config
.git/description
.git/hooks
.git/hooks/applypatch-msg.sample
.git/hooks/commit-msg.sample
.git/hooks/fsmonitor-watchman.sample
.git/hooks/post-update.sample
.git/hooks/pre-applypatch.sample
.git/hooks/pre-commit.sample
.git/hooks/pre-merge-commit.sample
.git/hooks/pre-push.sample
.git/hooks/pre-rebase.sample
.git/hooks/pre-receive.sample
.git/hooks/prepare-commit-msg.sample
.git/hooks/push-to-checkout.sample
.git/hooks/sendemail-validate.sample
.git/hooks/update.sample
.git/index
.gi

## Step 1. README와 `.gitignore` 만들기

실무에서는 저장소를 만들자마자 아래 두 가지를 먼저 정리하는 것이 좋습니다.

- `README.md`: 프로젝트 목적, 실행 방법, 구조 설명
- `.gitignore`: 추적하지 않을 파일 정의

특히 아래 항목은 보통 제외합니다.

- `.venv/`
- `__pycache__/`
- `.ipynb_checkpoints/`
- `.env`
- 시스템 캐시 파일

In [6]:
from pathlib import Path
import textwrap

readme = root / "README.md"
gitignore = root / ".gitignore"

readme.write_text(textwrap.dedent("""
# Week 3 - MLOps Infra: Code & Data Versioning

이 저장소는 Git + DVC 기반의 코드/데이터 버전관리 실습용 프로젝트입니다.

## 목적
- Git 기본 워크플로우 이해
- `.gitignore`와 README 작성
- DVC를 사용한 데이터 추적
- 데이터 버전 변경 및 복원 경험

## 권장 구조
- `data/`: raw / processed 데이터
- `notebooks/`: 분석용 노트북
- `src/`: Python 스크립트
- `reports/`: 시각화 및 결과물
- `.dvc/`: DVC 설정

## 핵심 명령
```bash
git init
dvc init
dvc add data/raw/customer_churn_sample.csv
git add .
git commit -m "feat: initialize Git and DVC structure"
```
""").strip() + "\n", encoding="utf-8")

gitignore.write_text(textwrap.dedent("""
# Python
__pycache__/
*.py[cod]
*.pyo

# Virtual environment
.venv/
venv/
env/

# Jupyter
.ipynb_checkpoints/

# Secrets / env
.env

# OS files
.DS_Store
Thumbs.db

# DVC cache
.dvc/cache/

# Optional large artifacts
models/
outputs/
""").strip() + "\n", encoding="utf-8")

print(readme.read_text(encoding="utf-8"))
print("-"*80)
print(gitignore.read_text(encoding="utf-8"))

# Week 3 - MLOps Infra: Code & Data Versioning

이 저장소는 Git + DVC 기반의 코드/데이터 버전관리 실습용 프로젝트입니다.

## 목적
- Git 기본 워크플로우 이해
- `.gitignore`와 README 작성
- DVC를 사용한 데이터 추적
- 데이터 버전 변경 및 복원 경험

## 권장 구조
- `data/`: raw / processed 데이터
- `notebooks/`: 분석용 노트북
- `src/`: Python 스크립트
- `reports/`: 시각화 및 결과물
- `.dvc/`: DVC 설정

## 핵심 명령
```bash
git init
dvc init
dvc add data/raw/customer_churn_sample.csv
git add .
git commit -m "feat: initialize Git and DVC structure"
```

--------------------------------------------------------------------------------
# Python
__pycache__/
*.py[cod]
*.pyo

# Virtual environment
.venv/
venv/
env/

# Jupyter
.ipynb_checkpoints/

# Secrets / env
.env

# OS files
.DS_Store
Thumbs.db

# DVC cache
.dvc/cache/

# Optional large artifacts
models/
outputs/



## Step 2. Git 저장소 초기화

### 핵심 명령

```bash
git init
git status
git add .
git commit -m "docs: initialize project structure"
```

In [7]:
import os, subprocess

os.chdir(root)

def run_live(cmd):
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    return result.returncode

run_live("git init")
run_live("git status")

$ git init
Reinitialized existing Git repository in /Users/sungjae-cha/Documents/03 아주대AI대학원/2026-1st-semester/ajou-mlops-2026-1nd-semester/week3-mlops-versioning/.git/

$ git status
On branch main
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/raw/customer_churn_sample 2.csv

nothing added to commit but untracked files present (use "git add" to track)



0

## Step 3. Git 사용자 정보 설정 확인

```bash
git config user.name
git config user.email
```

### 값이 비어 있으면 아래를 먼저 설정합니다. 
### 예시 name "smilesjcha"
### 예시 email "smilechacha@kaist.ac.kr"

```bash
git config --global user.name "Your Name"
git config --global user.email "your-email@example.com"
```

In [8]:
run_live("git config user.name")
run_live("git config user.email")
print("값이 비어 있다면 위 마크다운의 명령으로 먼저 설정하세요.")

$ git config user.name
smilesjcha

$ git config user.email
smilechacha@kaist.ac.kr

값이 비어 있다면 위 마크다운의 명령으로 먼저 설정하세요.


## Step 4. 첫 번째 commit 만들기

좋은 예:
- `docs: initialize project structure`
- `chore: add README and gitignore`

나쁜 예:
- `update`
- `final`
- `수정`

In [9]:
run_live("git add .")
run_live('git commit -m "docs: initialize project structure"')
run_live("git log --oneline --decorate -n 5")

$ git add .
$ git commit -m "docs: initialize project structure"
[main a3946ca] docs: initialize project structure
 1 file changed, 31 insertions(+)
 create mode 100644 data/raw/customer_churn_sample 2.csv

$ git log --oneline --decorate -n 5
a3946ca (HEAD -> main) docs: initialize project structure
3284c46 data: update churn sample dataset to v2
6ebf1b3 feat: track dataset with DVC
fdfeaf7 docs: initialize project structure



0

## Step 5. 샘플 데이터 생성

이번 노트북은 인터넷 없이도 실습할 수 있도록 간단한 CSV 데이터를 직접 생성합니다.  
이 데이터는 실제 모델링용이 아니라, **DVC tracking 실습용 예제 데이터**입니다.

In [10]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 30

df = pd.DataFrame({
    "customer_id": range(1001, 1001+n),
    "age": np.random.randint(20, 50, size=n),
    "avg_order_value": np.random.randint(20000, 200000, size=n),
    "purchase_count_30d": np.random.randint(0, 10, size=n),
    "is_member": np.random.choice([0, 1], size=n),
    "churned": np.random.choice([0, 1], size=n, p=[0.7, 0.3])
})

data_path = root / "data" / "raw" / "customer_churn_sample.csv"
df.to_csv(data_path, index=False, encoding="utf-8")
print("Saved:", data_path)
display(df.head())
print("Shape:", df.shape)

Saved: /Users/sungjae-cha/Documents/03 아주대AI대학원/2026-1st-semester/ajou-mlops-2026-1nd-semester/week3-mlops-versioning/data/raw/customer_churn_sample.csv


,customer_id,age,avg_order_value,purchase_count_30d,is_member,churned
0,1001,26,105305,1,0,0
1,1002,39,179765,3,1,1
2,1003,48,150608,6,1,1
3,1004,34,176730,7,1,0
4,1005,30,104478,2,0,0


Shape: (30, 6)


## Step 6. 왜 이 CSV를 Git에 바로 넣지 않고 DVC로 추적할까?

작은 예제 파일이라 Git에 그냥 넣어도 당장은 동작합니다.  
하지만 실무에서는 다음 문제가 생깁니다.

- 데이터가 커짐
- 바이너리/대용량 파일이 많아짐
- 어떤 데이터 버전으로 실험했는지 추적이 어려움
- Git clone/pull이 무거워짐

그래서 **실제 데이터는 DVC가 관리**하고, Git에는 `.dvc` 메타파일만 기록합니다.

## Step 7. DVC 초기화

```bash
dvc init
git status
```

In [11]:
run_live("dvc init")
run_live("git status")

$ dvc init
ERROR: failed to initiate DVC - '.dvc' exists. Use `-f` to force.

$ git status
On branch main
nothing to commit, working tree clean



0

## Step 8. 데이터 파일을 DVC로 추적하기

```bash
dvc add data/raw/customer_churn_sample.csv
```

실행 후 보통 다음 변화가 생깁니다.

- `customer_churn_sample.csv.dvc` 파일 생성
- `.gitignore`에 데이터 경로 추가
- Git은 데이터 본체 대신 메타파일을 추적

In [12]:
run_live("dvc add data/raw/customer_churn_sample.csv")
run_live("git status")

$ dvc add data/raw/customer_churn_sample.csv

To track the changes with git, run:

	git add data/raw/customer_churn_sample.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true

⠋ Checking graph


$ git status
On branch main
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   data/raw/customer_churn_sample.csv.dvc

no changes added to commit (use "git add" and/or "git commit -a")



0

## Step 9. `.dvc` 파일 확인하기

`.dvc` 파일 안에는 보통 아래 정보가 담깁니다.

- 추적 파일 경로
- 해시값
- 크기

즉, 실제 데이터가 아니라 **데이터 상태를 설명하는 포인터**입니다.

In [13]:
dvc_files = list((root / "data" / "raw").glob("*.dvc"))
print("DVC files:", dvc_files)
for fp in dvc_files:
    print(f"\n--- {fp.name} ---")
    print(fp.read_text(encoding="utf-8"))

DVC files: [PosixPath('/Users/sungjae-cha/Documents/03 아주대AI대학원/2026-1st-semester/ajou-mlops-2026-1nd-semester/week3-mlops-versioning/data/raw/customer_churn_sample.csv.dvc')]

--- customer_churn_sample.csv.dvc ---
outs:
- md5: 0035e53b22bf409adda6b79737217bff
  size: 687
  hash: md5
  path: customer_churn_sample.csv



## Step 10. Git에 무엇을 commit해야 하는가

### Git에 commit할 것
- `.dvc/`
- `.gitignore` 변경사항
- `*.dvc` 파일
- 코드, README 등 텍스트 파일

### Git에 commit하지 않을 것
- 실제 대용량 데이터 본체
- 캐시, 가상환경, 비밀키

In [14]:
run_live("git add .")
run_live('git commit -m "feat: track dataset with DVC"')
run_live("git log --oneline --decorate -n 10")

$ git add .
$ git commit -m "feat: track dataset with DVC"
[main a95976b] feat: track dataset with DVC
 1 file changed, 2 insertions(+), 2 deletions(-)

$ git log --oneline --decorate -n 10
a95976b (HEAD -> main) feat: track dataset with DVC
a3946ca docs: initialize project structure
3284c46 data: update churn sample dataset to v2
6ebf1b3 feat: track dataset with DVC
fdfeaf7 docs: initialize project structure



0

## Step 11. DVC remote 연결

실습에서는 두 가지 방식 중 하나를 사용할 수 있습니다.

### 옵션 A. 로컬 폴더 remote
- 가장 쉽고 빠름
- 클라우드 계정 필요 없음

### 옵션 B. 클라우드/원격 remote
- S3
- Azure Blob
- Google Drive
- SSH 서버

이 노트북에서는 **재현 가능한 로컬 실습**을 위해 로컬 폴더 remote를 사용합니다.

In [15]:
dvc_remote_dir = root.parent / "week3_dvc_remote_storage"
dvc_remote_dir.mkdir(parents=True, exist_ok=True)
print("DVC remote directory:", dvc_remote_dir)

run_live(f'dvc remote add -d localstorage "{dvc_remote_dir.as_posix()}"')
run_live("dvc remote list")

DVC remote directory: /Users/sungjae-cha/Documents/03 아주대AI대학원/2026-1st-semester/ajou-mlops-2026-1nd-semester/week3_dvc_remote_storage
$ dvc remote add -d localstorage "/Users/sungjae-cha/Documents/03 아주대AI대학원/2026-1st-semester/ajou-mlops-2026-1nd-semester/week3_dvc_remote_storage"
Setting 'localstorage' as a default remote.

ERROR: unexpected error - 'ascii' codec can't encode characters in position 101-106: ordinal not in range(128)

Having any troubles? Hit us up at https://dvc.org/support, we are always happy to help!

$ dvc remote list


0

## Step 12. 실제 데이터 push

```bash
dvc push
```

이 단계가 끝나면 다른 사람은 Git 메타정보를 받은 뒤 `dvc pull`로 같은 데이터를 다시 가져올 수 있습니다.

In [16]:
run_live("dvc push")

$ dvc push
ERROR: failed to push data to the cloud - config file error: no remote specified in /Users/sungjae-cha/Documents/03 아주대AI대학원/2026-1st-semester/ajou-mlops-2026-1nd-semester/week3-mlops-versioning. Create a default remote with
    dvc remote add -d <remote name> <remote url>



1

## Step 13. GitHub 원격 저장소 연결(선택)

```bash
git branch -M main
git remote add origin https://github.com/<YOUR_ID>/<YOUR_REPO>.git
git push -u origin main
```

또는 SSH:

```bash
git remote add origin git@github.com:<YOUR_ID>/<YOUR_REPO>.git
git push -u origin main
```

In [17]:
print("아래 명령은 예시입니다. 실제 저장소 URL로 바꿔서 터미널에서 실행하세요.")
print('git branch -M main')
print('git remote add origin https://github.com/<YOUR_ID>/<YOUR_REPO>.git')
print('git push -u origin main')

아래 명령은 예시입니다. 실제 저장소 URL로 바꿔서 터미널에서 실행하세요.
git branch -M main
git remote add origin https://github.com/<YOUR_ID>/<YOUR_REPO>.git
git push -u origin main


## Step 14. 데이터 버전 변경 실습

이번에는 같은 파일을 수정해 **데이터 v2**를 만들어봅니다.

예시 변경:
- 새 row 추가
- 일부 값 수정

In [18]:
df_v2 = pd.read_csv(data_path)

new_rows = pd.DataFrame([
    {"customer_id": 2001, "age": 31, "avg_order_value": 85000, "purchase_count_30d": 4, "is_member": 1, "churned": 0},
    {"customer_id": 2002, "age": 27, "avg_order_value": 48000, "purchase_count_30d": 1, "is_member": 0, "churned": 1},
])

df_v2.loc[0, "avg_order_value"] = int(df_v2.loc[0, "avg_order_value"] * 1.1)
df_v2 = pd.concat([df_v2, new_rows], ignore_index=True)
df_v2.to_csv(data_path, index=False, encoding="utf-8")

display(df_v2.tail())
print("Updated shape:", df_v2.shape)

,customer_id,age,avg_order_value,purchase_count_30d,is_member,churned
27,1028,21,194073,4,0,0
28,1029,47,174969,7,1,0
29,1030,40,88148,9,0,0
30,2001,31,85000,4,1,0
31,2002,27,48000,1,0,1


Updated shape: (32, 6)


## Step 15. 변경된 데이터 다시 DVC 추적

```bash
dvc add data/raw/customer_churn_sample.csv
git add .
git commit -m "data: update churn sample dataset to v2"
dvc push
```

In [19]:
run_live("dvc add data/raw/customer_churn_sample.csv")
run_live("git status")
run_live("git add .")
run_live('git commit -m "data: update churn sample dataset to v2"')
run_live("dvc push")
run_live("git log --oneline --decorate -n 10")

$ dvc add data/raw/customer_churn_sample.csv

To track the changes with git, run:

	git add data/raw/customer_churn_sample.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true

⠋ Checking graph


$ git status
On branch main
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   data/raw/customer_churn_sample.csv.dvc

no changes added to commit (use "git add" and/or "git commit -a")

$ git add .
$ git commit -m "data: update churn sample dataset to v2"
[main 9191d93] data: update churn sample dataset to v2
 1 file changed, 2 insertions(+), 2 deletions(-)

$ dvc push
ERROR: failed to push data to the cloud - config file error: no remote specified in /Users/sungjae-cha/Documents/03 아주대AI대학원/2026-1st-semester/ajou-mlops-2026-1nd-semester/week3-mlops-versioning. Create a default remote with
    dvc remote add -d <remote name> <remote url>

$ git

0

## Step 16. 버전 비교 포인트 정리

이제 저장소에는 최소 2개의 의미 있는 상태가 존재합니다.

1. 프로젝트 구조 초기화
2. 초기 데이터 버전 DVC tracking
3. 데이터 v2 반영

이 상태에서 질문할 수 있어야 합니다.

- 어떤 commit이 데이터 v1과 연결되는가?
- 어떤 commit이 데이터 v2와 연결되는가?
- 과거 상태로 복원하려면 무엇을 해야 하는가?

In [20]:
run_live("git log --oneline --decorate --graph --all -n 20")

$ git log --oneline --decorate --graph --all -n 20
* 9191d93 (HEAD -> main) data: update churn sample dataset to v2
* a95976b feat: track dataset with DVC
* a3946ca docs: initialize project structure
* 3284c46 data: update churn sample dataset to v2
* 6ebf1b3 feat: track dataset with DVC
* fdfeaf7 docs: initialize project structure



0

## Step 17. 과거 버전 복원 실습

### 복원 원리
1. 원하는 Git commit으로 이동
2. 해당 시점의 `.dvc` 메타정보 확인
3. `dvc pull` 실행
4. 그 시점의 데이터 상태 복원

In [61]:
print("먼저 아래 명령으로 commit 목록을 확인하세요:")
print("git log --oneline")
print("\n예시:")
print("git checkout <COMMIT_HASH>")
print("dvc pull")
print("\n주의: detached HEAD 상태가 될 수 있으므로, 복원 후 다시 main 브랜치로 돌아오세요.")

먼저 아래 명령으로 commit 목록을 확인하세요:
git log --oneline

예시:
git checkout <COMMIT_HASH>
dvc pull

주의: detached HEAD 상태가 될 수 있으므로, 복원 후 다시 main 브랜치로 돌아오세요.


## Step 18. 복원 데모용 보조 셀

아래 셀은 **실제 복원 명령을 자동 실행하지 않고**, 사용자가 원하는 커밋 해시를 넣어 실행하도록 만들어 둔 셀입니다.

In [62]:
TARGET_COMMIT = ""  # 예: "a1b2c3d"

if TARGET_COMMIT.strip():
    run_live(f"git checkout {TARGET_COMMIT}")
    run_live("dvc pull")
    print("\n현재 데이터 미리보기:")
    import pandas as pd
    restored = pd.read_csv("data/raw/customer_churn_sample.csv")
    display(restored.head())
    print("Shape:", restored.shape)
else:
    print("TARGET_COMMIT 값이 비어 있습니다. 복원하려는 commit hash를 넣고 다시 실행하세요.")

TARGET_COMMIT 값이 비어 있습니다. 복원하려는 commit hash를 넣고 다시 실행하세요.


## Step 19. 현재 브랜치로 복귀하기

```bash
git switch main
dvc pull
```

또는 Git 버전에 따라:

```bash
git checkout main
dvc pull
```

In [63]:
print("복원 실습 후 사용할 예시 명령:")
print("git switch main")
print("dvc pull")

복원 실습 후 사용할 예시 명령:
git switch main
dvc pull


## Step 20. 실습 중 체크해야 할 상태 확인 명령

```bash
git status
git log --oneline --graph --decorate -n 10
dvc status
dvc remote list
```

In [21]:
run_live("git status")
run_live("dvc status")
run_live("dvc remote list")

$ git status
On branch main
nothing to commit, working tree clean

$ dvc status
Data and pipelines are up to date.

$ dvc remote list


0

## Step 21. GitHub 제출용 확인 항목

- GitHub 저장소 링크 제출 가능 여부
- `README.md` 존재
- 최소 2개 이상의 의미 있는 commit 존재
- `.dvc` 파일 존재
- `dvc remote list` 결과 확인 가능
- 데이터 변경 전/후 차이를 설명 가능
- `Git만으로 부족한 이유`를 서술 가능

## Step 22. 자주 발생하는 오류와 해결 방법

### 1) `fatal: unable to auto-detect email address`
원인: Git 사용자 정보 미설정  
해결:
```bash
git config --global user.name "Your Name"
git config --global user.email "your-email@example.com"
```

### 2) `remote origin already exists`
원인: 이미 원격 저장소가 등록됨  
해결:
```bash
git remote -v
git remote remove origin
git remote add origin <NEW_URL>
```

### 3) GitHub 인증 실패(403)
원인:
- 비밀번호 인증 사용
- 권한 없는 저장소
- PAT/SSH 미설정

해결:
- HTTPS 대신 **PAT** 사용
- 또는 SSH key 등록 후 SSH URL 사용

### 4) 실제 데이터 파일이 Git에 들어감
원인: `dvc add` 전에 `git add .` 수행  
해결:
```bash
git rm --cached data/raw/customer_churn_sample.csv
```
이후 `dvc add` 재실행

### 5) `dvc push` 실패
원인:
- remote 경로 오류
- 권한 문제
- remote 미설정

해결:
```bash
dvc remote list
dvc doctor
```

### 6) `dvc pull` 후 데이터가 안 내려옴
원인:
- 해당 버전 데이터가 remote에 없음
- 아직 `dvc push` 안 함
- 잘못된 commit으로 checkout

해결:
- 먼저 데이터를 올린 환경에서 `dvc push`
- 올바른 commit인지 `git log` 확인

## Step 23. 실무 해석 포인트

다음 질문에 답할 수 있어야 합니다.

1. 왜 `final_v2_real_last.csv` 같은 파일명 방식은 위험한가?
2. 왜 Git만으로는 ML 실험 재현성이 부족한가?
3. DVC는 어떤 점에서 Git LFS보다 ML 친화적인가?
4. 팀 협업 시 `git pull + dvc pull` 구조가 왜 중요한가?

## Step 24. 간단 회고 작성

1. Git만으로 부족했던 이유를 2~3문장으로 설명하세요.
2. DVC를 도입하면 협업에서 무엇이 좋아지는지 설명하세요.
3. 이번 실습에서 가장 헷갈렸던 지점은 무엇이었나요?
4. 이후 MLflow / Serving / Monitoring과 어떤 식으로 연결될 것 같나요?

In [65]:
reflection_template = """
[회고 초안]

1) Git만으로 부족했던 이유
- 

2) DVC가 필요한 이유
- 

3) 협업 관점에서 좋아지는 점
- 

4) 실습 중 어려웠던 점
- 
"""
print(reflection_template)


[회고 초안]

1) Git만으로 부족했던 이유
- 

2) DVC가 필요한 이유
- 

3) 협업 관점에서 좋아지는 점
- 

4) 실습 중 어려웠던 점
- 



## Step 25. 선택 확장 실습

- `processed/` 데이터도 따로 DVC tracking 하기
- `src/preprocess.py` 스크립트 작성 후 Git commit 나누기
- `feature/...`, `exp/...` branch 만들어보기
- GitHub PR 흐름 실습
- DVC remote를 로컬 폴더 대신 S3 / Azure Blob으로 바꾸기
- `dvc repro`, `dvc stage add`로 파이프라인 추적 확장하기

## 최종 요약

1. **Git은 코드와 메타정보의 역사**를 관리한다.
2. **DVC는 데이터와 실험 입력의 역사**를 관리한다.
3. 머신러닝의 재현성은 **Git + DVC가 함께 작동할 때** 비로소 강해진다.

이제 여러분은 단순히 파일을 저장한 것이 아니라,  
**팀이 다시 복원할 수 있는 ML 실험 구조**를 하나 만든 것입니다.